# Hydrological Ensemble Verification with Scoreflow

This notebook demonstrates how to perform hydrometeological forecast verification using Scoreflow. It walks through setting up a verification pipeline, running it on NetCDF input data, and exploring the results interactively.

Forecast verification is essential for understanding how well model predictions match observations and how forecast skill changes with lead time. In this example, we compare multiple forecast variants against observed discharge and explore the results using interactive visualiozation.

By the end of this notebook, you will:
- Understand how to configure a verification pipeline
- Run Scoreflow on local (NetCDF) data
- Inspect and interpret the results
- Explore forecast performance using interactive visualizations

## Dataset and experiment setup

This example uses one observation dataset (**obs**) and three forecast datasets stored as NetCDF files.

We compare the following forecast variants against observations:

- **raw_raw**: baseline forecast
- **lin_log**: forecast with linear-log transformation
- **qqt_qqt**: forecast using quantile transformation

The verification is performed over a defined time period and evaluated across multiple forecast lead times (forecast periods).

## Imports and environment setup

We start by importing the required libraries and Scoreflow components.  
This includes configuration objects, datasource definitions, and scoring methods.

In [9]:
# Add automatic reloading of modules in case of changes
%load_ext autoreload
%autoreload 2
from datetime import datetime, timezone
from pathlib import Path

from app import create_app

from dpyverification.configuration import Config, GeneralInfoConfig
from dpyverification.configuration.utils import (
    ForecastPeriods,
    Range,
    TimeUnits,
    VerificationPair,
    VerificationPeriod,
)
from dpyverification.constants import DataType
from dpyverification.datasources import NetCDFConfig
from dpyverification.pipeline import run_pipeline
from dpyverification.scores import CrpsForEnsembleConfig, RankHistogramConfig

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Defining the verification configuration

Scoreflow uses a configuration-driven approach to define the verification pipeline.

The configuration is divided into three main components:

- **Datasources**: where the input data comes from  
- **General settings**: how the verification is structured  
- **Scores**: which metrics are computed  

We define each of these components step by step below.

### Data sources (input data)

Datasources define where Scoreflow retrieves the data used in verification.

In this example, we use:
- one observation dataset (`obs`)
- three forecast datasets (`raw_raw`, `lin_log`, `qqt_qqt`)

Each dataset is stored as a NetCDF file and loaded using a datasource configuration.

These datasets are assumed to follow the Scoreflow-compatible data model, which the pipeline expects in terms of dimensions and coordinates.

The datasets used in this notebook were obtaine from .........

In [ ]:
# Import NetCDF data from from local storage 
# Date origin: a Zarr archive
data_dir = Path.cwd().parent / "data"

### General verification settings

The general configuration defines how the verification is performed.

This includes:
- the **verification period** (time window for evaluation)
- the **forecast periods** (lead times)
- the **verification pairs** (which forecasts are compared against observations)

Each verification pair represents a comparison between one forecast dataset and the observations.

In [12]:
# start time = '1990-03-13T00:00:00.000000000'
# end time = '2010-07-31T00:00:00.000000000'
general = GeneralInfoConfig(
    verification_period=VerificationPeriod(
        start=datetime(1990, 3, 1, tzinfo=timezone.utc),
        end=datetime(2010, 7, 31, tzinfo=timezone.utc),
        dimension="forecast_reference_time",
    ),
    verification_pairs=[
        VerificationPair(id="raw_raw", obs="obs", sim="raw_raw"),
        VerificationPair(id="lin_log", obs="obs", sim="lin_log"),
        VerificationPair(id="qqt_qqt", obs="obs", sim="qqt_qqt"),
    ],
    forecast_periods=ForecastPeriods(unit=TimeUnits.day, values=Range(start=0, end=10, step=1)),
)

## Verification scores

Scores define how forecast performance is evaluated.

In this example, we compute:

- **Continuous Ranked Probability Score (CRPS)**  
  A measure of probabilistic forecast accuracy. Lower values indicate better performance.

- **Rank Histogram**  
  A diagnostic tool used to assess ensemble calibration and spread.

These scores are computed for each verification pair and across forecast periods.

In [14]:
config = Config(
    fileversion="0.1.0",
    general=general,
    datasources=[
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="obs",
            data_type=DataType.observed_historical,
            directory=str(data_dir),
            filename_glob="obs_Q.nc",
        ),
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="lin_log",
            data_type=DataType.simulated_forecast_ensemble,
            directory=str(data_dir),
            filename_glob="case-lin-log_Q.nc",
        ),
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="raw_raw",
            data_type=DataType.simulated_forecast_ensemble,
            directory=str(data_dir),
            filename_glob="case-raw-raw_Q.nc",
        ),
        NetCDFConfig(
            general=general,
            import_adapter="netcdf",
            source="qqt_qqt",
            data_type=DataType.simulated_forecast_ensemble,
            directory=str(data_dir),
            filename_glob="case-qqt-qqt_Q.nc",
        ),
    ],
    scores=[
        # Don't reduce any dimensions for the CRPS, as we want to be able to analyze it along
        # all dimensions (e.g. forecast_reference_time)
        CrpsForEnsembleConfig(score_adapter="crps_for_ensemble", general=general, reduce_dims=[]),
        # Reduce the forecast_reference_time dimension for the rank histogram, as we want to analyze
        # the overall rank histogram over the whole verification period
        RankHistogramConfig(
            score_adapter="rank_histogram",
            general=general,
            reduce_dims=["forecast_reference_time"],
        ),
    ],
)

## Running the verification pipeline

Once the configuration is defined, the pipeline can be executed with a single command.

Scoreflow will:
1. Load and align the input datasets
2. Apply any required transformations
3. Compute the requested scores
4. Return a standardized output object

The result is stored in an `OutputDataset`, which contains all verification results.

In [15]:
output_dataset = run_pipeline(config)

## Inspecting the output

The pipeline returns a Python `OutputDataset`, which provides a standardized interface to the verification results. This makes it possible to inspect results programmatically, compare verification pairs, and pass the output directly into custom visualization tools.

The `OutputDataset` contains:
- The defined verification pairs
- The computed scores
- The aligned datasets used in verification

Each verification pair represents a comparison between one forecast and the observations.

In [ ]:
type(output_dataset)

In [21]:
output_dataset.verification_pairs

[VerificationPair(id='raw_raw', obs='obs', sim='raw_raw'),
 VerificationPair(id='lin_log', obs='obs', sim='lin_log'),
 VerificationPair(id='qqt_qqt', obs='obs', sim='qqt_qqt')]

In [23]:
list(ds.data_vars)
list(ds.coords)

['station',
 'variable',
 <StandardDim.forecast_reference_time: 'forecast_reference_time'>,
 <StandardDim.forecast_period: 'forecast_period'>,
 'units',
 'lat',
 'lon',
 'realization',
 'time',
 'rank']

## Exploring a single verification result

We can extract results for a specific verification pair and inspect the underlying data.

The returned dataset is an `xarray.Dataset`, which allows easy access to:
- forecast values
- observations
- computed scores
- coordinates such as station, variable, and forecast period

In [17]:
ds = output_dataset.get(verification_pair=config.general.verification_pairs[2])
ds

<xarray.Dataset> Size: 144MB
Dimensions:                  (station: 88, variable: 1,
                              forecast_reference_time: 2919,
                              forecast_period: 10, realization: 5, rank: 6)
Coordinates:
  * station                  (station) <U11 4kB 'H-RN-0001' ... 'H-RN-WURZ'
    lat                      (station) float64 704B 0.0 0.0 0.0 ... 0.0 0.0 0.0
    lon                      (station) float64 704B 0.0 0.0 0.0 ... 0.0 0.0 0.0
  * variable                 (variable) <U1 4B 'Q'
    units                    (variable) <U1 4B '-'
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 23kB 19...
  * forecast_period          (forecast_period) timedelta64[ns] 80B 1 days ......
    time                     (forecast_reference_time, forecast_period) datetime64[ns] 234kB ...
  * realization              (realization) int64 40B 0 1 2 3 4
  * rank                     (rank) float64 48B 1.0 2.0 3.0 4.0 5.0 6.0
Data variables:
    obs                      (variable, station, forecast_reference_time, forecast_period) float64 21MB ...
    qqt_qqt                  (variable, forecast_reference_time, forecast_period, station, realization) float64 103MB ...
    crps_for_ensemble        (variable, forecast_reference_time, forecast_period, station) float64 21MB ...
    histogram_rank           (variable, station, forecast_period, rank) float64 42kB ...
Attributes:
    data_type:  observed_historical

## Interactive visualization

To explore the results, we use an interactive Dash application.

This app allows you to:
- Compare different forecast variants
- Select stations and variables
- Explore results across forecast lead times

Two main views are available:

### Scatter plot
Shows observed vs forecast values:
- Each point represents a forecast vs observation pair
- The 1:1 line indicates perfect agreement

### CRPS plot
Shows forecast skill over time:
- Lower values indicate better performance
- Helps compare forecast quality across lead times

In [8]:
app = create_app(output_dataset)
app.run(jupyter_mode="inline", jupyter_height=1400)

## Interpretation of results

The visualizations help assess forecast performance:

### Scatter plot
- Points close to the diagonal line indicate good agreement
- Spread of points reflects forecast uncertainty
- Bias can be observed as systematic deviation from the line

### CRPS
- Lower CRPS values indicate better forecast skill
- Increasing CRPS with lead time is expected
- Differences between methods show which approach performs better

### Rank histogram
- A flat histogram indicates a well-calibrated ensemble
- U-shaped distributions suggest under-dispersion
- Dome-shaped distributions suggest over-dispersion

These diagnostics provide insight into both accuracy and reliability of the forecasts.

## Optional: Using YAML-based configuration instead of Python 

In this notebook, we used Python objects to define the configuration.

Scoreflow also supports YAML-based configuration files, which:
- improve reproducibility
- make pipelines easier to share
- simplify deployment in production systems

Both approaches produce identical results.